        # ⛺ P1　營地任務：資料清理與特徵工程
        **統計冒險之旅 2026**　｜　Day 3（09/24 四）🗻 預測之巔　｜　關卡　｜　🏅 100 XP

        📖 實作；資料：勇者咖啡八月訂單（原始匯出版）
        　從「原始匯出版」到能用的特徵表

        ### 🎯 這一關你會學到
        - 把「原始匯出版」的八月訂單清乾淨：重複、缺值、名稱不一致、日期格式、異常值
- 算出每位會員的 RFM 特徵並 merge 回會員表
- 存成新的特徵表給後面的模型用

        ### 🧭 闖關方式
        1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
        2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
        3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
        4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/stats-quest-2026/)。

        > 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。
        > 🎲 這門課的答案常常是小數：任務會告訴你要把答案存進哪個變數，檢查時允許小小的誤差；切分、抽樣、模型請照題目用 `random_state=42`。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  統計冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins, math, warnings
warnings.filterwarnings("ignore")

_LEVEL = "P1"
_COURSE_NAMESPACE = "stats-quest-2026-datama"
_PREFIX = "SQ"
_TASKS = ["P1-1", "P1-2", "P1-3", "P1-4", "P1-5", "P1-6"]
_XP_EACH = 16
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_sq_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

# ---------------- 判分器 2.0 ----------------
class _Miss(Exception):
    pass

def 抓變數(ns, name, 型別=None):
    """從任務格執行後的變數取值；沒有就給友善訊息。"""
    if name not in ns:
        raise _Miss(f"我找不到變數 {name}，請確認你有把答案存進名字叫 {name} 的變數（大小寫要一樣）。")
    v = ns[name]
    if 型別 is not None and not isinstance(v, 型別):
        raise _Miss(f"{name} 的型別看起來不對（目前是 {type(v).__name__}）。")
    return v

def _num(v):
    try:
        import numpy as _np
        if hasattr(v, "item"): v = v.item()
    except Exception:
        pass
    return float(v)

def 約等於(v, 目標, 容差=None, 相對=0.01):
    """數值容差：|v-目標| <= 容差（預設為 目標 的 1%，且至少 1e-9）"""
    try:
        x = _num(v)
    except Exception:
        return False
    if x != x:   # NaN
        return False
    tol = 容差 if 容差 is not None else max(abs(目標) * 相對, 1e-9)
    return abs(x - 目標) <= tol

def 資料框像(obj, 列=None, 欄=None, 含欄位=None, 種類="DataFrame"):
    """檢查 DataFrame / Series：列數、欄數、必須包含的欄位；回傳 (ok, 訊息)"""
    import pandas as _pd
    if 種類 == "DataFrame" and not isinstance(obj, _pd.DataFrame):
        return False, f"這應該是一個 DataFrame（目前是 {type(obj).__name__}）。"
    if 種類 == "Series" and not isinstance(obj, _pd.Series):
        return False, f"這應該是一個 Series（目前是 {type(obj).__name__}）。"
    if 列 is not None and len(obj) != 列:
        return False, f"列數應該是 {列}，目前是 {len(obj)}。"
    if 欄 is not None and getattr(obj, "shape", (0, 0))[1] != 欄:
        return False, f"欄數應該是 {欄}，目前是 {obj.shape[1]}。"
    if 含欄位:
        cols = list(obj.columns) if hasattr(obj, "columns") else list(obj.index)
        missing = [c for c in 含欄位 if c not in cols]
        if missing:
            return False, "缺少欄位：" + "、".join(map(str, missing))
    return True, ""

class _NeedMoreInput(Exception):
    pass

_BUILTIN_NAMES = ("sum", "list", "dict", "set", "str", "int", "float", "max", "min", "len",
                  "print", "type", "range", "sorted", "abs", "round", "tuple", "map", "filter",
                  "open", "format", "all", "any", "zip", "bool", "next", "chr", "ord", "id")

_HIST = builtins.__dict__.setdefault("_sq_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_sq_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_sq_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

_CALL = re.compile(r"\s*(檢查|通關密語|全部檢查)\s*\(")

def _clean_cell(cell):
    return "\n".join(ln for ln in cell.splitlines() if not _CALL.match(ln))

def _is_mine(cell):
    s = cell.strip()
    if not s:
        return False
    if "#@title" in s or "任務定義(" in s or "_sq_" in s:
        return False
    if _CALL.match(s):
        return False
    return True

def _find_cells(tid):
    marker = "# 🎯 任務 " + tid
    marked = free = None
    im = ifree = -1
    for i, cell in enumerate(_history()):
        if not _is_mine(cell):
            continue
        if marker in cell:
            marked, im = cell, i
        elif "🎯 任務" not in cell:
            free, ifree = cell, i
    return marked, im, free, ifree

def _describe(src):
    body = [ln for ln in src.splitlines() if ln.strip() and not ln.strip().startswith("#")]
    if not body:
        return "（空白）"
    first = body[0].strip()
    return ("%s%s（共 %d 行）" % (first[:52], "…" if len(first) > 52 else "", len(body)))

def _fig_info(_plt):
    out = []
    try:
        for n in _plt.get_fignums():
            f = _plt.figure(n)
            for ax in f.get_axes():
                out.append(dict(title=ax.get_title() or "", xlabel=ax.get_xlabel() or "", ylabel=ax.get_ylabel() or "",
                                n_lines=len(ax.lines), n_patches=len(ax.patches), n_collections=len(ax.collections),
                                legend=bool(ax.get_legend())))
    except Exception:
        pass
    return out

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        run.shadowed = []
        for _n in _BUILTIN_NAMES:
            _b = getattr(builtins, _n, None)
            if _n in ns and _b is not None and ns[_n] is not _b:
                ns.pop(_n, None)
                run.shadowed.append(_n)
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        run.figs = []
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                run.figs = _fig_info(_plt)
                _plt.show = _orig_show
                _plt.close("all")
        return buf.getvalue(), ns
    run.src = src
    run.figs = []
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _fix_shadowed():
    try:
        ns = get_ipython().user_ns
    except Exception:
        ns = globals()
    bad = []
    for n in _BUILTIN_NAMES:
        b = builtins.__dict__.get(n)
        if b is not None and n in ns and ns[n] is not b:
            del ns[n]
            bad.append(n)
    return bad

def _progress():
    done = 0
    total = 0
    for t in _TASKS:
        total += 1
        if _PASSED.get(t):
            done += 1
    bar = "■" * done + "□" * (total - done)
    return f"[{bar}] {done}/{total}"

def _run_check(tid, src):
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        return False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。", []
    except _Miss as e:
        return False, str(e), getattr(run, "shadowed", [])
    except Exception:
        tb = traceback.format_exc().strip().splitlines()[-1]
        return False, "程式執行時發生錯誤 → " + tb, getattr(run, "shadowed", [])
    ok, extra = (result, "") if isinstance(result, bool) else result
    return ok, extra, getattr(run, "shadowed", [])

def _pass(tid):
    first = not _PASSED.get(tid)
    _PASSED[tid] = True
    print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")

def 檢查(tid):
    _shadow = _fix_shadowed()
    tid = builtins.str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    marked, im, free, ifree = _find_cells(tid)
    if marked is None and free is None:
        print(f"❌ 這次執行階段裡，我找不到你寫的程式。")
        print(f"   👉 請先按「# 🎯 任務 {tid}」那一格左邊的 ▶ 執行它，再執行這一格。")
        print("   （如果剛剛重新啟動過執行階段，上面每一格都要重跑一次，包含最上面的魔法工具箱）")
        return
    order = []
    if marked is not None:
        order.append(("標記", marked))
    if free is not None and ifree > im:
        order.append(("最後執行", free))
    if not order:
        order = [("最後執行", free)]
    tried = []
    for kind, src in order:
        ok, extra, shadowed = _run_check(tid, _clean_cell(src))
        tried.append((kind, src, extra, shadowed))
        if ok:
            _pass(tid)
            if extra:
                print("   💬 " + str(extra))
            if _shadow:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(_shadow)} 拿來當變數名了，我已經幫你還原。")
                print("      建議換個名字（例如 total、items），不然後面的程式會出現很難懂的錯誤。")
            if kind == "最後執行":
                print(f"   ℹ️ 你的程式最上面少了「# 🎯 任務 {tid}」那一行，我是用你最後執行的那一格判分的。")
                print("      把那一行加回去，之後的檢查會更準確。")
            if shadowed:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(shadowed)} 拿來當變數名了，判分時我先幫你還原。")
            if all(_PASSED.get(t) for t in _TASKS):
                print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
            return
    kind, src, extra, shadowed = tried[0]
    print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
    if extra:
        print("   💬 " + str(extra))
    if _HINTS.get(tid):
        print("   💡 提示：" + _HINTS[tid])
    _sh = _shadow + [n for n in shadowed if n not in _shadow]
    if _sh:
        print(f"   ⚠️ 你把內建名稱 {'、'.join(_sh)} 拿來當變數名了（我已還原），這會造成很難懂的錯誤，請改名後重跑那一格。")
    print("   🔎 我判分的是這一段程式：" + _describe(src))
    print(f"      如果這不是你剛剛寫的版本 → 確認第一行的「# 🎯 任務 {tid}」有保留，並重新執行那一格，再按檢查。")

def 全部檢查():
    """出錯或重新啟動執行階段後，重跑完所有任務格，再用這個一次驗收整關。"""
    _fix_shadowed()
    print(f"🔁 重新檢查 {_LEVEL} 的 {len(_TASKS)} 個任務…")
    todo = []
    for t in _TASKS:
        marked, im, free, ifree = _find_cells(t)
        if marked is None and free is None:
            todo.append(t)
            continue
        檢查(t)
    if todo:
        print("⏭️ 這次還沒執行過的任務：" + "、".join(todo))
        print("   先按那幾格左邊的 ▶ 執行，再回來執行 全部檢查()。")

def 通關密語():
    _fix_shadowed()
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_COURSE_NAMESPACE}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：{_PREFIX}-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

try:
    import numpy as _np_, pandas as _pd_
    _np_.random.seed(42)
except Exception:
    pass
print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_P1_1(run):
    out, ns = run()
    if 抓變數(ns, "重複數") != 60: return (False, "重複數 = raw.duplicated().sum()。")
    if 抓變數(ns, "缺金額數") != 45: return (False, "缺金額數 = raw['金額'].isna().sum()。")
    if 抓變數(ns, "分店種類數") != 12: return (False, "分店種類數 = raw['分店'].nunique()。")
    return (抓變數(ns, "負數量數") == 18, "負數量數 = (raw['數量'] <= 0).sum()。")
任務定義("P1-1", _check_P1_1, 提示="nunique() 數有幾種；(raw['數量'] <= 0).sum() 數有幾筆。")

def _check_P1_2(run):
    out, ns = run()
    d = 抓變數(ns, "df2")
    ok, msg = 資料框像(d, 列=2581)
    if not ok: return (False, msg + " 先 drop_duplicates()。")
    return (sorted(d["分店"].unique().tolist()) == ["中壢店", "信義店", "板橋店"], "分店應只剩三種：去空白（含全形空白）後，沒有「店」字的補上。")
任務定義("P1-2", _check_P1_2, 提示="df2['分店'].where(df2['分店'].str.endswith('店'), df2['分店'] + '店')。")

def _check_P1_3(run):
    out, ns = run()
    import pandas as _pd
    d = 抓變數(ns, "df3")
    ok, msg = 資料框像(d, 列=2563)
    if not ok: return (False, msg + " 數量 <= 0 的列要拿掉。")
    if not _pd.api.types.is_datetime64_any_dtype(d["日期"]): return (False, "日期 欄要是 datetime 型別（pd.to_datetime）。")
    return (str(d["日期"].min().date()) == "2026-08-01" and str(d["日期"].max().date()) == "2026-08-31", "日期範圍應為 2026-08-01～2026-08-31；三種格式都要先整理成 YYYY-MM-DD。")
任務定義("P1-3", _check_P1_3, 提示="df3[df3['數量'] > 0]。")

def _check_P1_4(run):
    out, ns = run()
    d = 抓變數(ns, "清理後")
    ok, msg = 資料框像(d, 列=2563, 含欄位=["金額", "單價", "會員編號"])
    if not ok: return (False, msg)
    if d["金額"].isna().any() or d["單價"].isna().any(): return (False, "金額、單價不可有缺值：金額 = 數量 × 單價。")
    if int(d["單價"].max()) > 200: return (False, "還有單價 9000 之類的錯字，用 標準單價 蓋回去。")
    return (約等於(抓變數(ns, "金額總和"), 360880, 0.5), "金額總和應為 360,880。")
任務定義("P1-4", _check_P1_4, 提示="清理後['數量'] * 清理後['單價']。")

def _check_P1_5(run):
    out, ns = run()
    d = 抓變數(ns, "rfm")
    ok, msg = 資料框像(d, 列=989, 含欄位=["會員編號", "R", "F", "M"])
    if not ok: return (False, msg + " 記得 reset_index()，且要先 dropna(subset=['會員編號'])。")
    if int(d["F"].sum()) != 1557: return (False, "F 總和應為 1557（每筆訂單算一次）。")
    if not 約等於(d["M"].sum(), 216450, 1): return (False, "M = 該會員八月金額總和。")
    return (約等於(d["R"].mean(), 12.6006, 0.05), "R = (基準日 - 最後一次來店日).days。")
任務定義("P1-5", _check_P1_5, 提示="M=(\"金額\", \"sum\")。")

def _check_P1_6(run):
    out, ns = run()
    d = 抓變數(ns, "members_rfm")
    ok, msg = 資料框像(d, 列=2000, 欄=15, 含欄位=["R", "F", "M", "回購"])
    if not ok: return (False, msg + " 用 members.merge(rfm, on='會員編號', how='left')。")
    if d[["R", "F", "M"]].isna().any().any(): return (False, "R、F、M 不可有缺值：F、M 填 0，R 填 31。")
    if 抓變數(ns, "來過人數") != 989: return (False, "來過人數 = (F > 0).sum()。")
    return (約等於(d["R"].mean(), 21.9015, 0.05), "R 缺值要填 31。")
任務定義("P1-6", _check_P1_6, 提示="members_rfm['R'].fillna(31)。")


In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
raw = pd.read_csv("https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.2.0/data/coffee_sales_aug_raw.csv")
members = pd.read_csv("https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.2.0/data/coffee_members.csv")
print(raw.shape)
raw.head()

## ⛺ 營地任務：資料清理與特徵工程
前面每一關拿到的都是「洗好的」資料。真實世界不是這樣：老闆從 POS 系統匯出的八月訂單長這樣——
有人按了兩次匯出（重複列）、分店名稱手打得亂七八糟、日期格式三種、單價多打兩個 0、還混進了幾筆退貨（數量是負的）。

**資料科學家 80% 的時間都在做這件事。** 這一關把它練成反射動作：

| 髒法 | 症狀 | 解法 |
|---|---|---|
| 重複列 | 同一筆訂單出現兩次 | `drop_duplicates()` |
| 名稱不一致 | 「信義店」「信義店 」「信義」 | `str.strip()`、補字、`map` |
| 日期格式混雜 | `2026-08-05`、`2026/08/05`、`20260805` | 先統一字串再 `pd.to_datetime` |
| 幽靈訂單 | 數量 ≤ 0 | 條件篩選 |
| 打錯的數字 | 單價 9000 | 用「常見值」修正、重算金額 |
| 缺值 | 金額是 NaN | 能算就算回來（數量 × 單價） |

> 🧭 原則：**先盤點，再動手；每動一步就數一次列數**，才知道自己有沒有不小心砍掉好資料。乾淨版應該有 2563 筆、金額總和 360,880。

In [ ]:
print("列數：", len(raw))
print("重複列：", raw.duplicated().sum())
print("各欄缺值：", raw.isna().sum().to_dict())
print("分店名稱：", raw["分店"].unique().tolist())
print("日期長相：", raw["日期"].str.len().value_counts().to_dict())
print("數量範圍：", raw["數量"].min(), "～", raw["數量"].max(), "｜ 單價範圍：", raw["單價"].min(), "～", raw["單價"].max())

### 🎯 任務 P1-1　盤點問題

先別急著清。算出四個數字：`重複數`（`duplicated().sum()`）、`缺金額數`（金額為 NaN 的筆數）、`分店種類數`（`nunique()`）、`負數量數`（數量 ≤ 0 的筆數）。

**預期結果（範例）**
```
60 45 12 18
```

In [ ]:
# 🎯 任務 P1-1　盤點問題（請保留這一行）
重複數 = int(raw.duplicated().sum())
缺金額數 = int(raw["金額"].isna().sum())
分店種類數 = ???
負數量數 = ???
print(重複數, 缺金額數, 分店種類數, 負數量數)

In [ ]:
檢查("P1-1")   # ◀ 執行這一格，看看任務 P1-1 有沒有過關

## 1-2　去重與名稱標準化
分店名稱的髒法有三種：前後多空白（含全形空白「　」）、少了「店」字。處理順序：先去空白，再補字。
`Series.where(條件, 否則的值)`：條件成立留原值，不成立就換成後面的值。

In [ ]:
s = pd.Series(["信義店 ", " 板橋店", "中壢店　", "信義", "板橋"])
s2 = s.str.replace("　", "").str.strip()
print(s2.tolist())
print(s2.where(s2.str.endswith("店"), s2 + "店").tolist())

### 🎯 任務 P1-2　去重＋分店名稱標準化

把 `raw` 去掉重複列後存成 `df2`，再把 `分店` 清成只有三種（信義店、板橋店、中壢店）。`df2` 應剩 2581 筆（幽靈訂單還在，下一題處理）。

**預期結果（範例）**
```
2581 ['信義店', '板橋店', '中壢店']
```

In [ ]:
# 🎯 任務 P1-2　去重＋分店名稱標準化（請保留這一行）
df2 = raw.drop_duplicates().copy()
df2["分店"] = df2["分店"].str.replace("　", "").str.strip()
df2["分店"] = ???                         # 沒有「店」字的補上
print(len(df2), df2["分店"].unique().tolist())

In [ ]:
檢查("P1-2")   # ◀ 執行這一格，看看任務 P1-2 有沒有過關

## 1-3　日期統一、幽靈訂單出局
三種日期格式：`2026-08-05`、`2026/08/05`、`20260805`。策略：**先把字串整理成同一種，再交給 `pd.to_datetime`**。
- 斜線換成減號：`.str.replace("/", "-")`
- 8 碼純數字加上減號：正規表達式 `r"^(\d{4})(\d{2})(\d{2})$"` → `r"\1-\2-\3"`（`regex=True`）

In [ ]:
s = pd.Series(["2026-08-05", "2026/08/05", "20260805"])
s2 = s.str.replace("/", "-").str.replace(r"^(\d{4})(\d{2})(\d{2})$", r"\1-\2-\3", regex=True)
print(pd.to_datetime(s2).tolist())

### 🎯 任務 P1-3　日期統一＋移除數量 ≤ 0

從 `df2` 做出 `df3`：`日期` 轉成 datetime（三種格式都要吃），再把數量 ≤ 0 的幽靈訂單移除。`df3` 應剩 2563 筆，日期從 2026-08-01 到 2026-08-31。

**預期結果（範例）**
```
2563 2026-08-01 2026-08-31
```

In [ ]:
# 🎯 任務 P1-3　日期統一＋移除數量 ≤ 0（請保留這一行）
df3 = df2.copy()
日期字串 = df3["日期"].str.replace("/", "-").str.replace(r"^(\d{4})(\d{2})(\d{2})$", r"\1-\2-\3", regex=True)
df3["日期"] = pd.to_datetime(日期字串)
df3 = ???                                 # 只留數量 > 0
print(len(df3), df3["日期"].min().date(), df3["日期"].max().date())

In [ ]:
檢查("P1-3")   # ◀ 執行這一格，看看任務 P1-3 有沒有過關

## 1-4　打錯的單價與缺掉的金額
每個品項的單價本來就固定，所以「這個品項最常見的單價」就是標準答案（`mode()`）。用它把所有單價蓋回去，金額再用 `數量 × 單價` 重算——順便把 NaN 的金額補回來。

In [ ]:
標準單價 = df3.groupby("品項")["單價"].agg(lambda s: s.mode().iloc[0])
print(標準單價.to_dict())
print("單價不等於標準的筆數：", (df3["單價"] != df3["品項"].map(標準單價)).sum())

### 🎯 任務 P1-4　修單價、補金額 → 乾淨版

做出 `清理後`：用 `標準單價` 蓋掉 `單價`、金額改為 `數量 × 單價`（整數）。驗收：2563 筆、金額總和 `金額總和` = 360,880、沒有任何缺值。

**預期結果（範例）**
```
2563 360880 0
```

In [ ]:
# 🎯 任務 P1-4　修單價、補金額 → 乾淨版（請保留這一行）
標準單價 = df3.groupby("品項")["單價"].agg(lambda s: s.mode().iloc[0])
清理後 = df3.copy()
清理後["單價"] = 清理後["品項"].map(標準單價)
清理後["金額"] = ???
金額總和 = int(清理後["金額"].sum())
print(len(清理後), 金額總和, 清理後.isna().sum().sum())

In [ ]:
檢查("P1-4")   # ◀ 執行這一格，看看任務 P1-4 有沒有過關

## 1-5　RFM：三個數字看懂一位客人
行銷最常用的客戶特徵叫 **RFM**：
- **R**ecency 最近一次來是幾天前（以 2026-08-31 為基準日）
- **F**requency 這個月來了幾次（訂單筆數）
- **M**onetary 這個月花了多少錢

八月有會員編號的訂單只有一部分（沒登入會員的散客是 NaN，要先 `dropna`）。`groupby("會員編號").agg(...)` 一次算三個。

In [ ]:
基準日 = pd.Timestamp("2026-08-31")
會員訂單 = 清理後.dropna(subset=["會員編號"])
print("有會員編號的訂單：", len(會員訂單), "｜ 不同會員：", 會員訂單["會員編號"].nunique())
示範 = 會員訂單.groupby("會員編號").agg(最近一次=("日期", "max"), 次數=("訂單編號", "count")).head(3)
print(示範)

### 🎯 任務 P1-5　算 RFM

做出 `rfm`（欄位：會員編號、R、F、M）：R = 基準日 − 最後一次來店日的天數、F = 訂單筆數、M = 金額總和。應有 989 位會員，F 總和 1557。

**預期結果（範例）**
```
989 1557 216450
```

In [ ]:
# 🎯 任務 P1-5　算 RFM（請保留這一行）
基準日 = pd.Timestamp("2026-08-31")
會員訂單 = 清理後.dropna(subset=["會員編號"])
rfm = 會員訂單.groupby("會員編號").agg(
    R=("日期", lambda s: (基準日 - s.max()).days),
    F=("訂單編號", "count"),
    M=???,
).reset_index()
print(len(rfm), rfm["F"].sum(), rfm["M"].sum())
rfm.head()

In [ ]:
檢查("P1-5")   # ◀ 執行這一格，看看任務 P1-5 有沒有過關

## 1-6　把特徵接回會員表
`members` 有 2,000 位會員，但八月只有一部分來過。用 `merge(how="left")` 接回去，沒來的人 F、M 填 0、R 填 31（整個月沒來）。
這張 `members_rfm` 就是 P3 要用的特徵表。

### 🎯 任務 P1-6　merge 回會員表

做出 `members_rfm`：`members` 左接 `rfm`（`on="會員編號"`），F、M 缺值填 0，R 缺值填 31。應為 2000 列 × 15 欄、`來過人數`（F > 0 的人數）= 989。

**預期結果（範例）**
```
(2000, 15) 989 0
```

In [ ]:
# 🎯 任務 P1-6　merge 回會員表（請保留這一行）
members_rfm = members.merge(rfm, on="會員編號", how="left")
members_rfm["F"] = members_rfm["F"].fillna(0)
members_rfm["M"] = members_rfm["M"].fillna(0)
members_rfm["R"] = ???
來過人數 = int((members_rfm["F"] > 0).sum())
print(members_rfm.shape, 來過人數, members_rfm.isna().sum().sum())
members_rfm.to_csv("members_rfm.csv", index=False)   # 存起來（P3 會重算一次，不用擔心遺失）

In [ ]:
檢查("P1-6")   # ◀ 執行這一格，看看任務 P1-6 有沒有過關

## 🌟 進階挑戰（不計分）
1. 用 `清理後` 畫每家分店每天的營收折線圖，看看和 `coffee_daily.csv` 的八月是否吻合。
2. 把 RFM 各自切成 1～3 分（`pd.qcut`），合成「RFM 分數」，找出前 10% 的 VIP。
3. 想一想：如果重複列不是「整列相同」，而是同一筆訂單被輸入兩次但時間差了一秒，你會怎麼抓？

---
## 🔑 通關密語
　你已經會把髒資料變成能用的特徵表——這是所有模型的地基。
全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：⛺ P2 營地任務：時間序列與基準模型** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/stats-quest-2026/blob/v1.2.0/notebooks/P2_camp_timeseries.ipynb)

回到入口網頁：https://johnnychao.github.io/stats-quest-2026/